# APTOS 2019 — single-seed ordinal vs MSE research (Kaggle GPU)

This is a **research-only**, non-clinical notebook. One version trains **one model and one seed**. No API token belongs in any cell. Create separate private Kaggle kernel versions via `scripts/package_kaggle_experiment.py`. The attached Kaggle input is `mariaherrerot/aptos2019`; it is read-only. 

**Default is dry-run.** GPU training happens only in an explicitly generated bundle with `RUN_EXPERIMENTS = True`. No measured results are implied by these instructions.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

RUN_EXPERIMENTS = False  # Explicitly set by local bundle builder --run
MODE = "reg"  # "reg" or "ordinal", injected by bundle builder
SEED = 42  # Injected by bundle builder
KAGGLE_INPUT = Path("/kaggle/input/aptos2019")
REPO = Path("/kaggle/working/aptos-research")
REPO_URL = "https://github.com/senanurcetin/APTOS-2019-diabetic-retinopathy.git"

print(f"Research matrix: mode={MODE}, seed={SEED}, GPU run={RUN_EXPERIMENTS}")
if MODE not in ("reg", "ordinal") or not isinstance(SEED, int):
    raise ValueError("Invalid model mode or random seed")


## Dataset, repository and dependency preflight

Attach the official APTOS mirror in Kaggle notebook settings. With internet enabled, the notebook clones the public repository, records the commit and installs the training dependencies. Mounted original retinal PNGs are linked read-only into the expected historical split layout. The workflow fails if labels or image directories are incomplete or ambiguous.


In [ ]:
if RUN_EXPERIMENTS:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Kaggle GPU not available: enable a GPU accelerator first")
    if not KAGGLE_INPUT.is_dir():
        raise FileNotFoundError("Attach mariaherrerot/aptos2019 to this Kaggle notebook")
    if not (REPO / "configs" / "cv.yaml").is_file():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
    version = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    print("Pinned run source commit:", version)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"], cwd=REPO, check=True)
    subprocess.run([sys.executable, str(REPO / "scripts" / "kaggle_dataset_preflight.py"),
                    "--repo-root", str(REPO), "--input-root", str(KAGGLE_INPUT)], check=True)
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Dry run only. No dataset downloaded, GPU allocated or model trained.")


## Data integrity and full five-fold experiment

The stage runner prepares the labels, scans retinal files, builds the baseline cache and checks duplicates **before** training. Do not skip the quality/leakage stage. Each job preserves the untouched APTOS test holdout and saves five fold checkpoints, validation thresholds and image identities. Full 15-epoch configuration is fixed for MSE and ordinal models alike. A Kaggle run might hit resource or time limits; incomplete runs must be reported as incomplete.


In [ ]:
if RUN_EXPERIMENTS:
    subprocess.run([sys.executable, "-m", "aptos.pipeline", "run", "quality"], cwd=REPO, check=True)
    command = [sys.executable, "-m", "aptos.training.cv", "--config", "configs/cv.yaml",
               "--mode", MODE, "--seed", str(SEED), "--no-track"]
    print("Running:", " ".join(command))
    subprocess.run(command, cwd=REPO, check=True)
    import json
    completed = []
    for folder in sorted((REPO / "models" / "cv").glob("baseline-*")):
        config_file, summary_file = folder / "config.json", folder / "summary.json"
        if not config_file.is_file() or not summary_file.is_file():
            continue
        config = json.loads(config_file.read_text())
        summary = json.loads(summary_file.read_text())
        expected = int(config["cv.folds"])
        if config["train.mode"] == MODE and config["train.seed"] == SEED:
            ok = summary.get("folds") == expected and all(
                (folder / f"fold{i}.pt").is_file() and
                (folder / f"fold{i}.json").is_file() and
                (folder / f"fold{i}.npz").is_file()
                for i in range(1, expected + 1)
            )
            completed.append({"directory": folder.name, "complete": ok})
    if not completed or not all(item["complete"] for item in completed):
        raise RuntimeError("No fully completed CV sweep: do not claim successful training")
    print(json.dumps({"repository_commit": version, "mode": MODE, "seed": SEED,
                      "verified_sweeps": completed}, indent=2))
else:
    print("Data preparation and training not run. Generate bundle with --run --push.")


## After both model families complete

The two independent jobs must be returned to the **same research environment** with verified checkpoints before a paired analysis. Obtain output files with `kaggle kernels output`; compare only aligned authentic per-image predictions. For IDRiD and Messidor-2, attach both authorized external datasets, score each model with the APTOS-fitted thresholds, then run `aptos.evaluation.compare_sweeps`. See `docs/ordinal-benchmark.md` and keep all negative results.
